In [24]:
import torch
import torchvision.transforms as transforms
import torchvision
import torch.utils.data as dataloader
import torch.nn as nn

In [25]:
#transformation from PIL to tensor format
transformation_operation = transforms.Compose([transforms.ToTensor()])

In [26]:
 train_dataset = torchvision.datasets.MNIST(root='./data',
                                            train =True, download = True, transform= transformation_operation)
 val_dataset = torchvision.datasets.MNIST(root='./data', train =False , download=True, transform= transformation_operation)

In [27]:
#define variable
num_classes = 10
batch_size = 64
num_channels = 1
patch_size = 7
img_size =28
num_patches = (img_size//patch_size)**2
print(num_patches)
embedding_dim = 64
attention_heads = 4
transformer_block = 4
learning_rate = 0.001
epochs = 5
mlp_hidden_nodes = 128

16


In [28]:
#define dataset batches
train_loader = dataloader.DataLoader(train_dataset, batch_size = batch_size, shuffle = True)
val_loader = dataloader.DataLoader(val_dataset, batch_size = batch_size, shuffle = True)


In [29]:
#part1: patch embedding
#part2: Transformer encoder
#part3: MLP_head
#vision transformer class

In [47]:
#patch embedding
class PatchEmbedding(nn.Module):
  def __init__(self):
    super().__init__()
    self.patch_embed = nn.Conv2d(num_channels, embedding_dim, kernel_size= 7, stride= patch_size)
  def forward(self, x):
    #patch_embedding
    x =self.patch_embed(x)
    #flattening
    x = x.flatten(2)
    x = x.transpose(1,2)
    return x

In [48]:
#transformer incoder
class TransformerEncoder(nn.Module):
  def __init__(self):
    super().__init__()
    self.layer_norm1 = nn.LayerNorm(embedding_dim)
    self.layer_norm2= nn.LayerNorm(embedding_dim)
    self.multihead_attention = nn.MultiheadAttention(embedding_dim, attention_heads, batch_first=True)
    self.mlp = nn.Sequential(
        nn.Linear(embedding_dim, mlp_hidden_nodes),
        nn.GELU(),
        nn.Linear(mlp_hidden_nodes,embedding_dim),
    )
  def forward(self, x):
    residual1 = x
    x = self.layer_norm1(x)
    x = self.multihead_attention(x,x,x)[0]
    x = x +residual1

    residual2 = x
    x=self.layer_norm2(x)
    x=self.mlp(x)
    x = x+residual2
    return x

In [49]:
class MLP_head(nn.Module):
  def __init__(self):
    super().__init__()
    self.layer_norm1 = nn.LayerNorm(embedding_dim)
    self.mlp_head = nn.Linear(embedding_dim, num_classes)
  def forward(self,x ):
    x =self.layer_norm1(x)
    x=self.mlp_head(x)

    return x


In [50]:
class VisionTransformer(nn.Module):
  def __init__(self):
    super().__init__()
    self.patch_embedding = PatchEmbedding()
    self.cls_token = nn.Parameter(torch.randn(1,1,embedding_dim))
    self.positional_embed = nn.Parameter(torch.randn(1,1+num_patches, embedding_dim))
    self.transformer_block = nn.Sequential(*[TransformerEncoder()for _ in range(transformer_block)])
    self.mlp_head = MLP_head()

  def forward(self,x):
    x = self.patch_embedding(x)
    B = x.size(0)
    class_tokens = self.cls_token.expand(B, -1, -1)
    x = torch.cat((class_tokens, x), dim =1)  #dim = 0 batch
    x = x + self.positional_embed
    x = self.transformer_block(x)
    x = x[:,0]
    x = self.mlp_head(x)
    return x

In [51]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = VisionTransformer().to(device)
optim = torch.optim.Adam(model.parameters(), lr= learning_rate)
criterion = nn.CrossEntropyLoss()

In [52]:
for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    correct_epoch = 0
    total_epoch = 0

    print(f"\nEpoch {epoch+1}")

    for batch_idx, (images, labels) in enumerate(train_loader):
        images = images.to(device)
        labels = labels.to(device)

        optim.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optim.step()

        train_loss += loss.item()

        preds = outputs.argmax(dim=1)
        correct = (preds == labels).sum().item()
        accuracy = 100.0 * correct / labels.size(0)

        correct_epoch += correct
        total_epoch += labels.size(0)

        if batch_idx % 10 == 0:
            print(f"Batch {batch_idx+1:3d}: Loss = {loss.item():.4f}, Accuracy = {accuracy:.2f}%")

    epoch_acc = 100.0 * correct_epoch / total_epoch
    avg_loss = train_loss / len(train_loader)

    print(f"==> Epoch {epoch+1} Summary: Avg Loss = {avg_loss:.4f}, Accuracy = {epoch_acc:.2f}%")


Epoch 1
Batch   1: Loss = 2.5711, Accuracy = 3.12%
Batch  11: Loss = 2.3466, Accuracy = 10.94%
Batch  21: Loss = 2.2683, Accuracy = 15.62%
Batch  31: Loss = 2.2092, Accuracy = 21.88%
Batch  41: Loss = 1.9079, Accuracy = 37.50%
Batch  51: Loss = 1.4657, Accuracy = 46.88%
Batch  61: Loss = 0.9633, Accuracy = 68.75%
Batch  71: Loss = 0.9831, Accuracy = 71.88%
Batch  81: Loss = 0.6702, Accuracy = 82.81%
Batch  91: Loss = 0.7526, Accuracy = 81.25%
Batch 101: Loss = 0.7579, Accuracy = 70.31%
Batch 111: Loss = 0.4285, Accuracy = 87.50%
Batch 121: Loss = 0.5356, Accuracy = 84.38%
Batch 131: Loss = 0.7532, Accuracy = 76.56%
Batch 141: Loss = 0.3415, Accuracy = 93.75%
Batch 151: Loss = 0.3234, Accuracy = 92.19%
Batch 161: Loss = 0.4426, Accuracy = 84.38%
Batch 171: Loss = 0.5334, Accuracy = 79.69%
Batch 181: Loss = 0.5924, Accuracy = 89.06%
Batch 191: Loss = 0.3400, Accuracy = 90.62%
Batch 201: Loss = 0.2696, Accuracy = 92.19%
Batch 211: Loss = 0.3480, Accuracy = 90.62%
Batch 221: Loss = 0.4124

In [ ]:
data_point, label = next(iter(train_loader))
print("Shape of data point:", data_point.shape)
patch_embed= nn.Conv2d(num_channels,embedding_dim, kernel_size=patch_size, stride= patch_size)
X = patch_embed(data_point)
print(X.shape)
X_flatten = X.flatten(2)
print(X_flatten.transpose(1,2).shape)
#class token
